## *Recolha e Pré-processamento*

In [1]:
from Bio import Entrez
import pandas as pd

Entrez.email = "conhecimentolinguagem@gmail.com"

term = '("disease"[MeSH Terms]) AND ("symptom"[Title/Abstract] OR "treatment"[Title/Abstract]) AND ("2021"[Date - Publication] : "2025"[Date - Publication])'

# Pesquisa
handle = Entrez.esearch(db="pubmed", term=term, retmax=2)  # podes aumentar retmax
record = Entrez.read(handle)
ids = record["IdList"]

articles = []

for pmid in ids:
    fetch = Entrez.efetch(db="pubmed", id=pmid, rettype="xml")
    data = Entrez.read(fetch)
    
    article_data = data['PubmedArticle'][0]['MedlineCitation']['Article']
    
    title = article_data.get('ArticleTitle', '')
    
    # Abstract pode ter várias partes (<AbstractText> pode ser lista)
    abstract_text = ''
    if 'Abstract' in article_data and 'AbstractText' in article_data['Abstract']:
        abstract_parts = article_data['Abstract']['AbstractText']
        if isinstance(abstract_parts, list):
            # Junta todas as partes
            abstract_text = ' '.join([str(part) for part in abstract_parts])
        else:
            abstract_text = str(abstract_parts)
    
    articles.append({
        'pmid': pmid,
        'title': title,
        'abstract': abstract_text
    })

df = pd.DataFrame(articles)
df.to_csv("articles.csv", index=False)
print(df)

       pmid                                              title  \
0  41261598  Crowned dens syndrome combined with cervical d...   
1  41257749  Transient Headache and Neurological Deficits w...   

                                            abstract  
0  Crowned dens syndrome (CDS) is an unusual and ...  
1  The transient Headache and Neurological Defici...  


## *Extração de Entidades*

- Criar um ambiente virtual novo
- pip install scapy==3.7.4
- pip install scispacy==0.5.1
- Download de "en_ner_bc5cdr_md" em https://allenai.github.io/scispacy/
- pip install "location"


#### *Transformers*

In [2]:
import spacy
import re
import csv
from collections import defaultdict
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
import pandas as pd

# -------------------------------
# --- Modelo NER ---
# -------------------------------
model_names = ["d4data/biomedical-ner-all"]
ner_pipelines = []
for name in model_names:
    tokenizer = AutoTokenizer.from_pretrained(name)
    model = AutoModelForTokenClassification.from_pretrained(name)
    ner_pipelines.append(pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple"))

# -------------------------------
# --- SciSpaCy ---
# -------------------------------
nlp_spacy = spacy.load("en_ner_bc5cdr_md")  # Disease e Chemical

# -------------------------------
# --- DrugBank ---
# -------------------------------
drugbank_list = []
try:
    with open("drugbank_dataset.csv", newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            drugbank_list.append(row['Common name'].strip().lower())
            if row.get('Synonyms'):
                for syn in row['Synonyms'].split('|'):
                    syn = syn.strip().lower()
                    if syn:
                        drugbank_list.append(syn)
    drugbank_list = list(set(drugbank_list))
    drugbank_list.append("albumin")
    print(f"Carregados {len(drugbank_list)} termos do DrugBank.")
except FileNotFoundError:
    print("Aviso: 'drugbank_dataset.csv' não encontrado. A lista de tratamentos pode estar incompleta.")
    drugbank_list = []


# -------------------------------
# --- Lista de sintomas comuns ---
# -------------------------------
sintomas_comuns = {
    "fever", "cough", "fatigue", "headache", "nausea", "vomiting",
    "shortness of breath", "pain", "swelling", "stiffness", "dizziness",
    "chills", "sore throat", "diarrhea", "rash", "itching"
}

# -------------------------------
# --- Funções auxiliares ---
# -------------------------------
def unir_substrings(entidades_set):
    """Remove entidades redundantes ou sobrepostas, priorizando a mais longa."""
    entidades = sorted(entidades_set, key=lambda x: -len(x))
    final = set()
    for e in entidades:
        # Adiciona a entidade 'e' se nenhuma entidade 'f' já em 'final' for uma superstring de 'e'
        # E se 'e' não for uma substring de nenhuma entidade 'f' já em 'final'
        # (Esta lógica simplificada prioriza a entidade mais longa que aparece primeiro)
        if not any(e in f for f in final):
            # Remove quaisquer substrings de 'e' que já possam estar em 'final'
            final = {f for f in final if f not in e}
            final.add(e)
    return final

def fundir_entidades_adjacentes(entidades_set, texto_lower):
    """
    Tenta fundir entidades na lista se a sua combinação existir no texto.
    Ex: Se "fetal" e "hydrops" estiverem no set, e "fetal hydrops" estiver no texto,
    substitui os dois por "fetal hydrops".
    """
    # Usamos um loop 'while' para permitir fusões recursivas
    # (ex: "severe" + "fetal" -> "severe fetal", depois "severe fetal" + "hydrops" -> "severe fetal hydrops")
    while True:
        houve_fusao = False
        entidades_list = sorted(list(entidades_set), key=len, reverse=True)
        novas_entidades = set()
        entidades_a_remover = set()

        # Copia o set para poder iterar com segurança
        entidades_a_verificar = set(entidades_set)

        for e1 in entidades_a_verificar:
            for e2 in entidades_a_verificar:
                if e1 == e2:
                    continue
                
                # Evita re-processar entidades já marcadas para remoção
                if e1 in entidades_a_remover or e2 in entidades_a_remover:
                    continue

                # Tenta combinar "e1 e2"
                combined_phrase = f"{e1} {e2}"

                # Se a frase combinada existir no texto...
                if combined_phrase in texto_lower:
                    # Verifica se esta combinação já não é uma substring de algo maior
                    if not any(combined_phrase in f and combined_phrase != f for f in entidades_set):
                        print(f"    → Fusão: '{e1}' + '{e2}' -> '{combined_phrase}'")
                        novas_entidades.add(combined_phrase)
                        entidades_a_remover.add(e1)
                        entidades_a_remover.add(e2)
                        houve_fusao = True

        # Aplica as mudanças da passagem
        if houve_fusao:
            entidades_set.update(novas_entidades)
            entidades_set.difference_update(entidades_a_remover)
        else:
            # Se não houve fusões nesta passagem, o processo está completo
            break

    return entidades_set

# -------------------------------
# --- Função principal (Versão Otimizada) ---
# -------------------------------
def extrair_entidades(texto):
    texto_lower = texto.lower()
    entity_votes = defaultdict(list)  # entidade -> lista de labels

    # --- AJUSTE 1: FILTRO DE CONFIANÇA ---
    # Ignora deteções dos transformers com score abaixo de 50%
    # Isto resolve o problema do "premature" (score 0.31)
    CONFIDENCE_THRESHOLD = 0.5 

    print("\n[DEBUG] --- Transformers Ensemble ---")
    for ner in ner_pipelines:
        print(f"\nModelo: {ner.model.name_or_path}")
        resultados = ner(texto)
        for r in resultados:
            token = r["word"].lstrip("#").lower().strip()
            token = re.sub(r'[^\w\s-]', '', token).strip()
            token = re.sub(r'\s+', ' ', token).strip()
            score = r['score']

            # APLICA O FILTRO DE CONFIANÇA
            if token and score >= CONFIDENCE_THRESHOLD:
                entity_votes[token].append(r['entity_group'])
                print(f"  - {token}: {r['entity_group']} ({score:.2f})")
            elif token:
                # Imprime o que foi rejeitado
                print(f"  - {token}: {r['entity_group']} ({score:.2f}) -- REJEITADO (Score baixo)")


    print("\n[DEBUG] --- SciSpaCy ---")
    doc = nlp_spacy(texto)
    for ent in doc.ents:
        entidade = texto[ent.start_char:ent.end_char].lower().strip()
        entidade = re.sub(r'[^\w\s-]', '', entidade).strip()
        entidade = re.sub(r'\s+', ' ', entidade).strip()

        if not entidade:
            continue

        if ent.label_ == "DISEASE":
            entity_votes[entidade].append("DISEASE")
            print(f"  - {entidade} [DISEASE] adicionada pelo SciSpaCy")
        elif ent.label_ == "CHEMICAL":
            entity_votes[entidade].append("CHEMICAL")
            print(f"  - {entidade} [CHEMICAL] adicionada pelo SciSpaCy")

    # -------------------------------
    # --- Distribuição por tipo com votação ---
    # -------------------------------
    entidades = {"DOENÇA": set(), "SINTOMA": set(), "TRATAMENTO": set()}

    priority_map = {
        "Sign_symptom": "SINTOMA",
        "Symptom": "SINTOMA",
        "SIGN": "SINTOMA",
        "PROBLEM": "SINTOMA",
        "TREATMENT": "TRATAMENTO",
        "PROCEDURE": "TRATAMENTO",
        "DISEASE": "DOENÇA",
        "Therapeutic_procedure": "TRATAMENTO",
    }

    print("\n[DEBUG] --- Distribuição por tipo com votos e prioridade ---")
    for entidade, labels in entity_votes.items():
        mapped_labels = [priority_map.get(l) for l in labels if l in priority_map]
        if not mapped_labels:
            continue

        if entidade in sintomas_comuns:
            final_label = "SINTOMA"
        elif "SINTOMA" in mapped_labels:
            final_label = "SINTOMA"
        elif "TRATAMENTO" in mapped_labels:
            final_label = "TRATAMENTO"
        else:
            final_label = "DOENÇA"

        entidades[final_label].add(entidade)
        print(f"Entidade: {entidade}, Labels associadas: {labels}")
        print(f"  → Voto final: {final_label} para {entidade}")

    # -------------------------------
    # --- DrugBank reforça tratamentos ---
    # -------------------------------
    print("\n[DEBUG] --- DrugBank ---")
    for drug in drugbank_list:
        if re.search(r'\b' + re.escape(drug) + r'\b', texto_lower):
            entidades["TRATAMENTO"].add(drug)
            print(f"  → DRUGBANK TRATAMENTO: {drug}")

    # -------------------------------
    # --- Heurística de Conflito de Substring ---
    # -------------------------------
    print("\n[DEBUG] --- Corrigindo conflitos de substring (ex: Doença vs Tratamento) ---")
    doencas_a_remover = set()
    tratamentos_a_adicionar = set()

    for doenca in entidades["DOENÇA"]:
        for tratamento in entidades["TRATAMENTO"]:
            if tratamento in doenca:
                print(f"  → Conflito: DOENÇA '{doenca}' contém TRATAMENTO '{tratamento}'.")
                print(f"     → Movendo '{doenca}' para TRATAMENTO.")
                doencas_a_remover.add(doenca)
                tratamentos_a_adicionar.add(doenca)
                break 

    entidades["DOENÇA"].difference_update(doencas_a_remover)
    entidades["TRATAMENTO"].update(tratamentos_a_adicionar)

    # -------------------------------
    # --- Limpeza de Ruído (Filtro) ---
    # -------------------------------
    print("\n[DEBUG] --- Filtrando ruído (ex: <= 2 caracteres) ---")
    excecoes_curtas = {drug for drug in drugbank_list if len(drug) <= 2}
    for k in entidades:
        entidades_a_remover = set()
        for entidade in entidades[k]:
            if len(entidade) <= 2 and entidade not in excecoes_curtas:
                entidades_a_remover.add(entidade)
                print(f"  → Removendo ruído: {entidade} (Categoria: {k})")
        entidades[k] -= entidades_a_remover
        
    # -------------------------------
    # --- AJUSTE GENÉRICO DE SINTOMAS ---
    # -------------------------------
    print("\n[DEBUG] --- Removendo sintomas genéricos (ex: symptoms) ---")
    termos_genericos = {"symptoms", "signs", "issues", "problems", "complaints"}
    for termo in termos_genericos:
        if termo in entidades["SINTOMA"]:
            print(f"  → Removendo termo genérico: {termo}")
            entidades["SINTOMA"].discard(termo)
    
    # -------------------------------
    # --- HEURÍSTICA: Fundir Entidades Adjacentes ---
    # -------------------------------
    print("\n[DEBUG] --- Tentando fundir entidades adjacentes (ex: fetal hydrops) ---")
    for k in entidades:
        print(f"  → Verificando fusões em: {k}")
        entidades[k] = fundir_entidades_adjacentes(entidades[k], texto_lower)

    # -------------------------------
    # --- Limpeza final ---
    # -------------------------------
    print("\n[DEBUG] --- Limpeza final (unindo substrings) ---")
    for k in entidades:
        entidades[k] = unir_substrings(entidades[k])

    return {k: sorted(list(v)) for k, v in entidades.items()}

# --- Aplicar aos artigos ---
def entities_from_title_abstract(row):
    # Assegura que title e abstract não são nulos
    titulo = str(row.get('title', ''))
    abstract = str(row.get('abstract', ''))
    
    texto = f"{titulo} {abstract}"
    return extrair_entidades(str(texto))

    
df = pd.read_csv("articles.csv") 

if 'df' in locals():
    print(f"Processando {len(df)} artigos...")
    df["entities"] = df.apply(entities_from_title_abstract, axis=1)

    # --- Mostrar alguns resultados corrigido ---
    entities_data = []
    for i, row in df.iterrows():
        entities_data.append({
           "pmid": row["pmid"],
           "title": row["title"],
           "abstract": row["abstract"],
           "DOENÇA": ", ".join(row["entities"]["DOENÇA"]),
           "SINTOMA": ", ".join(row["entities"]["SINTOMA"]),
           "TRATAMENTO": ", ".join(row["entities"]["TRATAMENTO"])
        })

    entities_df = pd.DataFrame(entities_data)
    entities_df.to_csv("entities.csv", index=False)
    print("\n\n--- RESULTADO FINAL (entities.csv) ---")

    for i, row in entities_df.iterrows():
        print(f"\nPMID: {row['pmid']}")
        print(f"  DOENÇA: {row['DOENÇA'] or 'Nenhuma'}")
        print(f"  SINTOMA: {row['SINTOMA'] or 'Nenhuma'}")
        print(f"  TRATAMENTO: {row['TRATAMENTO'] or 'Nenhuma'}")

c:\Users\ferna\anaconda3\envs\CLProject\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use cpu
c:\Users\ferna\anaconda3\envs\CLProject\lib\site-packages\spacy\language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


Carregados 51212 termos do DrugBank.
Processando 2 artigos...

[DEBUG] --- Transformers Ensemble ---

Modelo: d4data/biomedical-ner-all
  - crowned dens: Disease_disorder (0.83)
  - cervical: Biological_structure (0.97)
  - disc: Disease_disorder (0.94)
  - crowned dens syndrome: Disease_disorder (0.60)
  - cds: Disease_disorder (0.99)
  - intense: Severity (0.75)
  - neck: Biological_structure (1.00)
  - pain: Sign_symptom (1.00)
  - intra: Biological_structure (0.97)
  - neck: Biological_structure (1.00)
  - acute: Detailed_description (0.67)
  - cervical: Biological_structure (0.98)
  - disc: Disease_disorder (0.95)
  - intra: Biological_structure (1.00)
  - anial: Biological_structure (0.94)
  - infection: Disease_disorder (0.78)
  - cds: Disease_disorder (0.99)
  - 80 - year - old: Age (0.96)
  - female: Sex (0.95)
  - severe: Severity (1.00)
  - suboccipital neck: Biological_structure (0.88)
  - pain: Sign_symptom (1.00)
  - movement: Diagnostic_procedure (0.50)
  - neck: Biologi

## *Extração de Relações*

In [1]:
# ===============================================
# SCRIPT COMPLETO (ABORDAGEM HÍBRIDA)
#
# SISTEMA 1: Ensemble 3 Modelos (BART, BERT, DeBERTa) c/ Veto
#            para pares T-D, T-S, S-D
# SISTEMA 2: Especialista DDI (ELECTRA)
#            para pares T-T (Tratamento-Tratamento)
# ===============================================
import spacy
from transformers import (
    pipeline, 
    AutoTokenizer, 
    AutoModelForSequenceClassification
)
import torch
import itertools
import pandas as pd
import warnings
from collections import Counter
import re

# Suprimir avisos
warnings.filterwarnings(
    "ignore", 
    message="Token indices sequence length is longer than the specified maximum sequence length.*"
)

# --- 1. CARREGAR MODELO DE SENTENÇAS ---
nlp_sent = spacy.blank("en")
nlp_sent.add_pipe("sentencizer")

# --- 2. CARREGAR TODOS OS 4 MODELOS ---
# Globais para guardar os modelos
re_classifier_bart = None
re_classifier_bert = None
re_classifier_deberta = None
re_model_ddi = None
re_tokenizer_ddi = None
device_ddi = None

def carregar_todos_modelos():
    """Carrega os 3 modelos do Ensemble + 1 modelo DDI."""
    global re_classifier_bart, re_classifier_bert, re_classifier_deberta
    global re_model_ddi, re_tokenizer_ddi, device_ddi
    
    if re_classifier_bart: # Evitar recarregar
        print("Modelos já carregados.")
        return

    # --- MODELOS DO ENSEMBLE (SISTEMA 1) ---
    print("Carregando Modelo 1 (Generalista - BART)...")
    re_classifier_bart = pipeline(
        "zero-shot-classification",
        model="facebook/bart-large-mnli",
        device=0 if torch.cuda.is_available() else -1 
    )

    print("Carregando Modelo 2 (Especialista NLI - PubMedBERT)...")
    MODEL_NAME_BERT = "pritamdeka/PubMedBERT-MNLI-MedNLI"
    tokenizer_bert = AutoTokenizer.from_pretrained(MODEL_NAME_BERT)
    re_classifier_bert = pipeline(
        "zero-shot-classification",
        model=MODEL_NAME_BERT,
        tokenizer=tokenizer_bert,
        device=0 if torch.cuda.is_available() else -1
    )

    print("Carregando Modelo 3 (Especialista NLI - DeBERTa)...")
    MODEL_NAME_DEBERTA = "cross-encoder/nli-deberta-v3-base"
    tokenizer_deberta = AutoTokenizer.from_pretrained(MODEL_NAME_DEBERTA)
    re_classifier_deberta = pipeline(
        "zero-shot-classification",
        model=MODEL_NAME_DEBERTA,
        tokenizer=tokenizer_deberta,
        device=0 if torch.cuda.is_available() else -1
    )
    
    # --- MODELO DDI (SISTEMA 2) ---
    print("Carregando Modelo 4 (Especialista DDI - ELECTRA)...")
    MODEL_NAME_DDI = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract"
    device_ddi = "cuda" if torch.cuda.is_available() else "cpu"
    re_tokenizer_ddi = AutoTokenizer.from_pretrained(MODEL_NAME_DDI)
    re_model_ddi = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME_DDI).to(device_ddi)
    print("Todos os 4 modelos carregados.")


# --- 3. FUNÇÕES DE CLASSIFICAÇÃO ---

def classify_relation_ensemble(e1, e2, sentence, classifier, confidence_threshold, custom_labels):
    """Função para os 3 modelos do Ensemble Zero-Shot."""
    hypotheses = [f"{e1} {label} {e2}" for label in custom_labels]
    
    try:
        max_len = classifier.tokenizer.model_max_length
        tokenized_sentence = classifier.tokenizer(sentence, truncation=False)["input_ids"]
        
        if len(tokenized_sentence) > max_len:
            truncated_ids = tokenized_sentence[:max_len-1] + [tokenized_sentence[-1]]
            premise = classifier.tokenizer.decode(truncated_ids, skip_special_tokens=True)
        else:
            premise = sentence
        
        result = classifier(premise, hypotheses, multi_label=False)
        best_label = result["labels"][0]
        best_score = result["scores"][0]
        
        if best_label == "is not related" or best_score < confidence_threshold:
            return None, best_score
        return best_label, best_score

    except Exception as e:
        print(f"Erro no pipeline (Ensemble): {e}\nFrase: {sentence}\n")
        return None, 0.0

def classify_relation_ddi(e1, e2, sentence, conf_threshold):
    """Função dedicada para o modelo DDI (Tratamento-Tratamento)."""
    
    # Formato de input comum para DDI: [CLS] frase [SEP] e1 [SEP] e2 [SEP]
    text = f"{sentence} [SEP] {e1} [SEP] {e2}"
    
    try:
        inputs = re_tokenizer_ddi(text, return_tensors="pt", truncation=True, max_length=512).to(device_ddi)
        outputs = re_model_ddi(**inputs)
        
        pred = torch.argmax(outputs.logits, dim=1).item()
        score = torch.softmax(outputs.logits, dim=1)[0, pred].item()
        
        # Mapear o ID da previsão para a etiqueta
        label = re_model_ddi.config.id2label[pred]
        
        # Só retornamos se NÃO for "falso" E se passar o limiar
        if label.upper() != "DDI-FALSE" and score >= conf_threshold:
            return label, score # ex: "DDI-EFFECT"
        else:
            return None, score

    except Exception as e:
        print(f"Erro no pipeline (DDI): {e}\nFrase: {sentence}\n")
        return None, 0.0

# --- 4. FUNÇÃO PRINCIPAL (LÓGICA HÍBRIDA) ---
def extrair_relacoes_hibrido(texto, entidades, limiares_ensemble, limiar_ddi):
    
    relacoes = []
    
    # --- Etiquetas para o Ensemble (Sistema 1) ---
    LABELS_TRATAMENTO = ["treats", "causes", "is a side effect of", "is not related"]
    LABELS_SINTOMA = ["is a symptom of", "is not related"]
    
    # --- Padronização ---
    if not isinstance(entidades, dict): return [] 
    entidades_padronizadas = {}
    for tipo, lista in entidades.items():
        if isinstance(lista, (list, set)):
            entidades_padronizadas[tipo] = [str(e).lower().strip() for e in lista]
        
    doc_sent = nlp_sent(texto)

    for sent in doc_sent.sents:
        sent_text_lower = sent.text.lower()
        ents_in_sent = {}
        for tipo, lista in entidades_padronizadas.items():
            ents_in_sent[tipo] = [e for e in lista if e in sent_text_lower]
        if sum(len(v) for v in ents_in_sent.values()) < 2: continue

        # --- Geração de Pares (Separados por Lógica) ---
        pares_trat_prob = [] # T-D, T-S (Para o Ensemble)
        pares_sint_doenca = [] # S-D (Para o Ensemble)
        pares_trat_trat = [] # T-T (Para o Especialista DDI)
        
        # Combina Doenças e Sintomas numa única lista de "Problemas"
        problemas = ents_in_sent.get("DOENÇA", []) + ents_in_sent.get("SINTOMA", [])
        
        pares_trat_prob += list(itertools.product(ents_in_sent.get("TRATAMENTO", []), problemas))
        pares_sint_doenca += list(itertools.product(ents_in_sent.get("SINTOMA", []), ents_in_sent.get("DOENÇA", [])))
        pares_trat_trat += list(itertools.product(ents_in_sent.get("TRATAMENTO", []), ents_in_sent.get("TRATAMENTO", [])))
        
        # --- LÓGICA 1: Processar Pares T-Problema (Ensemble c/ Veto) ---
        for e1, e2 in pares_trat_prob:
            if e1 == e2: continue
            
            rel_bart, score_bart = classify_relation_ensemble(
                e1, e2, sent.text, re_classifier_bart, limiares_ensemble['BART'], LABELS_TRATAMENTO
            )
            rel_bert, score_bert = classify_relation_ensemble(
                e1, e2, sent.text, re_classifier_bert, limiares_ensemble['BERT'], LABELS_TRATAMENTO
            )
            rel_deberta, score_deberta = classify_relation_ensemble(
                e1, e2, sent.text, re_classifier_deberta, limiares_ensemble['DEBERTa'], LABELS_TRATAMENTO
            )
            
            if rel_bert is None: continue # Veto do Especialista NLI (BERT)
            
            votos = [v for v in [rel_bart, rel_bert, rel_deberta] if v is not None]
            if len(votos) < 2: continue
            
            rel_final, num_votos = Counter(votos).most_common(1)[0]
            if num_votos >= 2 and rel_final == rel_bert:
                relacoes.append({
                    "Entity1": e1, "Entity2": e2, "Relation": rel_final,
                    "Details": f"Ensemble_Veto (BERT: {score_bert:.2f})",
                    "Sentence": sent.text
                })
        
        # --- LÓGICA 1 (cont.): Processar Pares S-D (Ensemble c/ Veto) ---
        for e1, e2 in pares_sint_doenca:
            if e1 == e2: continue
            
            rel_bart, score_bart = classify_relation_ensemble(
                e1, e2, sent.text, re_classifier_bart, limiares_ensemble['BART'], LABELS_SINTOMA
            )
            rel_bert, score_bert = classify_relation_ensemble(
                e1, e2, sent.text, re_classifier_bert, limiares_ensemble['BERT'], LABELS_SINTOMA
            )
            rel_deberta, score_deberta = classify_relation_ensemble(
                e1, e2, sent.text, re_classifier_deberta, limiares_ensemble['DEBERTa'], LABELS_SINTOMA
            )
            
            if rel_bert is None: continue # Veto
            
            votos = [v for v in [rel_bart, rel_bert, rel_deberta] if v is not None]
            if len(votos) < 2: continue
            
            rel_final, num_votos = Counter(votos).most_common(1)[0]
            if num_votos >= 2 and rel_final == rel_bert:
                relacoes.append({
                    "Entity1": e1, "Entity2": e2, "Relation": rel_final,
                    "Details": f"Ensemble_Veto (BERT: {score_bert:.2f})",
                    "Sentence": sent.text
                })
        
        # --- LÓGICA 2: Processar Pares T-T (Especialista DDI) ---
        for e1, e2 in pares_trat_trat:
            if e1 == e2: continue
            
            label_ddi, score_ddi = classify_relation_ddi(e1, e2, sent.text, limiar_ddi)
            
            if label_ddi: # Se não for None (ou seja, não é "DDI-FALSE" e passou o limiar)
                relacoes.append({
                    "Entity1": e1,
                    "Entity2": e2,
                    "Relation": label_ddi, # ex: "DDI-EFFECT"
                    "Details": f"DDI_Specialist ({score_ddi:.2f})",
                    "Sentence": sent.text
                })

    return relacoes

def str_para_lista(entidade_str):
    """
    Converte uma string do CSV em lista de entidades.
    - Remove espaços extras
    - Ignora strings vazias
    - Converte para lowercase (opcional)
    """
    if not entidade_str or pd.isna(entidade_str):
        return []
    return [item.strip().lower() for item in entidade_str.split(",") if item.strip() != ""]

# --- 5. PONTO DE ENTRADA PRINCIPAL ---
def main():
    # --- Carregar todos os 4 modelos ---
    carregar_todos_modelos()
    
    # --- Carregar os seus dados ---
    df = pd.read_csv("entities.csv")
    
    # ---------------------------
    # Converter colunas de entidades (strings) para listas
    # ---------------------------
    df["DOENÇA"] = df["DOENÇA"].apply(str_para_lista)
    df["SINTOMA"] = df["SINTOMA"].apply(str_para_lista)
    df["TRATAMENTO"] = df["TRATAMENTO"].apply(str_para_lista)
    
    # ---------------------------
    # Construir o dicionário 'entities' que o script espera
    # ---------------------------
    df["entities"] = df.apply(lambda row: {
        "DOENÇA": row["DOENÇA"],
        "SINTOMA": row["SINTOMA"],
        "TRATAMENTO": row["TRATAMENTO"]
    }, axis=1)
    
    print(df["entities"].iloc[1])

    # Combinar title e abstract numa única coluna 'text'
    df['text'] = df['title'].fillna('') + " " + df['abstract'].fillna('')
   

    # --- Definir Limiares para os dois sistemas ---
    LIMIARES_ENSEMBLE = {
        'BART': 0.6,    # Relaxado
        'BERT': 0.35,   # Relaxado (O nosso "Veto")
        'DEBERTa': 0.55 # Relaxado
    }
    LIMIAR_DDI = 0.8 # Especialista DDI pode ser mais rigoroso

    print(f"\nExtraindo relações com SISTEMA HÍBRIDO...")
    df["relations"] = df.apply(
        lambda row: extrair_relacoes_hibrido(
            str(row["title"]) + " " + str(row["abstract"]), 
            row["entities"],
            LIMIARES_ENSEMBLE,
            LIMIAR_DDI
        ),
        axis=1
    )

    df.to_csv("relations.csv", index=False)
    print("Relações gravadas em 'relations.csv'")

    # --- Ver os resultados ---
    print("\nResultados Finais:")
    for idx, row in df.iterrows():
        print(f"--- Artigo {idx+1} ---")
        if row['relations']:
            for rel in row['relations']:
                print(f"  {rel['Entity1']} --[{rel['Relation']} ({rel['Details']})]--> {rel['Entity2']}")
        else:
            print("  Nenhuma relação encontrada.")

# Executar o script
if __name__ == "__main__":
    main()

c:\Users\ferna\anaconda3\envs\CLProject\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Carregando Modelo 1 (Generalista - BART)...


Device set to use cpu


Carregando Modelo 2 (Especialista NLI - PubMedBERT)...


Device set to use cpu


...

Carregando Modelo 4 (Especialista DDI - ELECTRA)...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Todos os 4 modelos carregados.
{'DOENÇA': ['aphasia', 'asthenia', 'autoimmune encephalitis', 'dysarthria', 'epilepsia', 'handl syndrome', 'hemiplegic migraine', 'neurological deficits', 'paresthesia', 'primary headache', 'proteinorrhachia', 'stroke', 'tumors', 'vasculitis'], 'SINTOMA': ['esthesia', 'headache', 'impaired speech', 'mig', 'motor symptoms', 'nausea', 'neurological deficit', 'orrh', 'par', 'raine', 'sensory symptoms'], 'TRATAMENTO': []}

Extraindo relações com SISTEMA HÍBRIDO...


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Relações gravadas em 'relations.csv'

Resultados Finais:
--- Artigo 1 ---
  pain --[pain is a symptom of crowned dens syndrome (Ensemble_Veto (BERT: 1.00))]--> crowned dens syndrome
  pain --[pain is a symptom of chronic neck pain (Ensemble_Veto (BERT: 1.00))]--> chronic neck pain
  pain --[pain is a symptom of herniation (Ensemble_Veto (BERT: 0.95))]--> herniation
  pain --[pain is a symptom of degenerative cervical spondylosis (Ensemble_Veto (BERT: 1.00))]--> degenerative cervical spondylosis
  pain --[pain is a symptom of fracture (Ensemble_Veto (BERT: 0.94))]--> fracture
  pain --[pain is a symptom of tumors (Ensemble_Veto (BERT: 1.00))]--> tumors
  calcium --[calcium is a symptom of knee joints (Ensemble_Veto (BERT: 1.00))]--> knee joints
  celecoxib --[celecoxib treats fever (Ensemble_Veto (BERT: 0.88))]--> fever
  celecoxib --[celecoxib treats pain (Ensemble_Veto (BERT: 0.98))]--> pain
  colchicine --[colchicine treats pain (Ensemble_Veto (BERT: 0.35))]--> pain
  pain --[pain is

## *Criação do Grafo*

In [21]:
import pandas as pd
import ast
from neo4j import GraphDatabase

# -----------------------------
# 1. CONFIGURAÇÃO DO NEO4J
# -----------------------------
URI = "neo4j://127.0.0.1:7687"
USER = "neo4j"
PASSWORD = "password"   # <-- coloca aqui a tua password


# -----------------------------
# 2. FUNÇÃO PARA CRIAR RELAÇÃO
# -----------------------------
def create_relation_in_neo4j(tx, entity1, type1, relation, entity2, type2):
    """
    Cria nós e relações no Neo4j com labels específicos:
    (:Doenca), (:Sintoma), (:Tratamento)
    """
    relation_type = relation.upper().replace(" ", "_").replace("-", "_")

    query = (
        f"MERGE (e1:{type1} {{nome: $entity1}})\n"
        f"MERGE (e2:{type2} {{nome: $entity2}})\n"
        f"MERGE (e1)-[:{relation_type}]->(e2)"
    )

    tx.run(query, entity1=entity1, entity2=entity2)


# -----------------------------
# 3. PROGRAMA PRINCIPAL
# -----------------------------
def main_neo4j():

    # --- 3.1 Ler ficheiros ---
    try:
        df_relations = pd.read_csv("relations.csv")
        df_entities = pd.read_csv("entities.csv")
    except FileNotFoundError as e:
        print(f"Erro: ficheiro '{e.filename}' não encontrado.")
        return

    # ----------------------------------------------
    # 3.2 Criar mapa {entidade: tipo} corretamente
    # ----------------------------------------------
    entity_to_type = {}

    for _, row in df_entities.iterrows():
        
        # 1. Tratar a coluna 'DOENÇA'
        diseases_str = str(row["DOENÇA"])
        if diseases_str and diseases_str.strip().lower() != 'nan' and diseases_str.strip().lower() != 'none':
            # Divide a string em itens pela vírgula
            diseases = [d.strip().lower() for d in diseases_str.split(',')]
            for d in diseases:
                if d:
                    entity_to_type[d] = "Doenca"
        
        # 2. Tratar a coluna 'SINTOMA'
        symptoms_str = str(row["SINTOMA"])
        if symptoms_str and symptoms_str.strip().lower() != 'nan' and symptoms_str.strip().lower() != 'none':
            symptoms = [s.strip().lower() for s in symptoms_str.split(',')]
            for s in symptoms:
                if s:
                    entity_to_type[s] = "Sintoma"

        # 3. Tratar a coluna 'TRATAMENTO'
        treatments_str = str(row["TRATAMENTO"])
        if treatments_str and treatments_str.strip().lower() != 'nan' and treatments_str.strip().lower() != 'none' and treatments_str.strip().lower() != 'nenhuma':
            treatments = [t.strip().lower() for t in treatments_str.split(',')]
            for t in treatments:
                if t:
                    entity_to_type[t] = "Tratamento"


    # -----------------------------
    # 3.3 Conectar ao Neo4j
    # -----------------------------
    try:
        driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
        driver.verify_connectivity()
        print("Conectado ao Neo4j com sucesso.")
    except Exception as e:
        print(f"Erro ao conectar ao Neo4j: {e}")
        return

    total_relations = 0
    unique_relations = set()

    with driver.session() as session:

        print("Limpando o grafo existente...")
        session.run("MATCH (n) DETACH DELETE n")

        # Processar relações
        for _, row in df_relations.iterrows():

            relations_raw = row.get("relations", None)
            if pd.isna(relations_raw):
                continue

            try:
                relations_list = ast.literal_eval(relations_raw)
            except Exception:
                continue

            for rel in relations_list:

                entity1 = rel["Entity1"].strip().lower()
                entity2 = rel["Entity2"].strip().lower()
                relation = rel["Relation"].strip().lower()

                # ⚠️ Se não existir no mapeamento, não importa
                if entity1 not in entity_to_type:
                    print(f"[IGNORADO] Entidade desconhecida: {entity1}")
                    continue
                if entity2 not in entity_to_type:
                    print(f"[IGNORADO] Entidade desconhecida: {entity2}")
                    continue

                type1 = entity_to_type[entity1]
                type2 = entity_to_type[entity2]

                # Evitar duplicados
                key = (entity1, relation, entity2)
                if key in unique_relations:
                    continue

                unique_relations.add(key)

                session.execute_write(
                    create_relation_in_neo4j,
                    entity1, type1,
                    relation,
                    entity2, type2
                )

                total_relations += 1

    driver.close()

    print("\n--------------------------------")
    print("✅ Importação concluída!")
    print(f"Total de relações inseridas: {total_relations}")
    print("--------------------------------")


# Executar
main_neo4j()


Conectado ao Neo4j com sucesso.
Limpando o grafo existente...

--------------------------------
✅ Importação concluída!
Total de relações inseridas: 41
--------------------------------
